# Phase 4.2 — SandboxRaceEnv Reward Function Validation

Validates the Phase 4.2 reward formula across four scenarios.

**Reward formula:**
```
per-step: position_reward (0.5 per overtake)
        + pit_cost        (-0.05 per pit attempt)
        + cliff_penalty   (-0.10 × laps_past_cliff)
        + invalid_penalty (-2.0 if same-compound pit)
        + pace_reward     ((rival_lap_time - ego_lap_time) × 0.05)  ← dense per-lap signal

terminal: -final_position × 2.0   # P1→-2, P10→-20
        + (10.0 if rule_met else -100.0)
```

**pace_reward notes:**
- Compares ego total lap time (incl. 22s pit if pitting) vs rival baseline lap time for that lap
- Rival baseline: pace_s1 before rival pit lap, pace_s1 + 22 on rival pit lap, pace_s2 after
- Circuit-agnostic: works at Monaco/Singapore where position changes are rare
- Scale 0.05 ≈ ±0.025–0.1 reward/lap for a ±0.5–2s pace advantage

**Cliff laps (from compound_constants.py):**
- SOFT: cliff at tire_age 18, 0.15 s/lap (lap time), 0.10 reward/lap
- MEDIUM: cliff at tire_age 32, 0.10 s/lap, 0.10 reward/lap
- HARD: cliff at tire_age 45, 0.06 s/lap, 0.10 reward/lap

**Note:** Monaco 2024 has a data quality issue (all stint_number=2 due to red-flagged start).
Scenario C uses Monaco 2023 — LEC on pole, 78 laps, valid for the same strategic test.

In [ ]:
import sys
sys.path.insert(0, '../backend/src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from pitiq.envs.sandbox import SandboxRaceEnv

print('SandboxRaceEnv imported OK')

In [ ]:
COMPOUND_ACTION = {'SOFT': 1, 'MEDIUM': 2, 'HARD': 3}


def run_strategy(env, config, pit_schedule, description=''):
    """Run a fixed strategy: pit_schedule maps lap_number → target compound."""
    obs, _ = env.reset(options=config)
    total_reward = 0.0
    done = False
    last_info = {}

    while not done:
        lap = env._lap_num
        if lap in pit_schedule:
            action = COMPOUND_ACTION[pit_schedule[lap]]
        else:
            action = 0

        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        done = terminated or truncated
        last_info = info

    log = env.reward_logger.to_dataframe()
    return {
        'description': description,
        'total_reward': total_reward,
        'final_position': last_info.get('position', '?'),
        'violated_rule': last_info.get('violated_two_compound_rule', False),
        'log': log,
    }


def summarise(results):
    """Print a ranked summary table and reward component breakdown."""
    rows = []
    for r in results:
        log = r['log']
        rows.append({
            'Strategy': r['description'],
            'Total Reward': round(r['total_reward'], 2),
            'Final Pos': r['final_position'],
            'Rule Violated': r['violated_rule'],
            'Pace Reward': round(log['pace_reward'].sum(), 3),
            'Pos Reward': round(log['position_reward'].sum(), 3),
            'Pit Cost': round(log['pit_cost'].sum(), 3),
            'Cliff Pen': round(log['cliff_penalty'].sum(), 3),
            'Invalid Pen': round(log['invalid_action_penalty'].sum(), 3),
            'Term Pos': round(log['terminal_pos_reward'].sum(), 2),
            'Term Rule': round(log['terminal_rule_reward'].sum(), 2),
        })
    df = pd.DataFrame(rows).sort_values('Total Reward', ascending=False).reset_index(drop=True)
    with pd.option_context('display.max_columns', 20, 'display.width', 220):
        print(df.to_string())
    return df


env = SandboxRaceEnv(log_rewards=True)
print('env created with reward logging enabled')

---
## Scenario A — Monza (Italian GP) 2025, VER on pole, MEDIUM start

**Circuit profile:** rival pace ~81.5 s, rivals pit at lap 34 (training median).

| Strategy | Pit stops | Expected behaviour |
|---|---|---|
| 1 | M→H lap 32 | Optimal 1-stop, 2 laps ahead of rivals |
| 2 | None | Rule violator (1 dry compound only) |
| 3 | M→S lap 18, S→H lap 36 | 2-stop, no cliff |
| 4 | 4-stop (M→S→M→H→S) | Wasteful, lots of pit costs + position drops |
| 5 | M→H lap 50 | Tire abuse: 18 laps past MEDIUM cliff (lap 32) |

**Expected assertion: 1 > 3 > 4 ≈ 5 > 2**
*(4 and 5 are approximately tied: 4-stop collapses to P20 via position loss, tire abuse has -17.1 cliff penalty but holds P13 — both effects roughly cancel)*

In [ ]:
MONZA_CFG = {
    'circuit': 'Italian Grand Prix',
    'driver': 'VER',
    'year': 2025,
    'total_laps': 53,
    'starting_position': 1,
    'starting_compound': 'MEDIUM',
    'two_compound_rule_enforced': True,
}

monza_strategies = [
    ({32: 'HARD'},                          'S1: Optimal 1-stop M→H L32'),
    ({},                                    'S2: No pit (rule violator)'),
    ({18: 'SOFT', 36: 'HARD'},              'S3: 2-stop M→S→H L18+36'),
    ({13: 'SOFT', 25: 'MEDIUM', 37: 'HARD', 47: 'SOFT'}, 'S4: Wasteful 4-stop'),
    ({50: 'HARD'},                          'S5: Tire abuse M→H L50'),
]

monza_results = []
for pit_sched, desc in monza_strategies:
    r = run_strategy(env, MONZA_CFG, pit_sched, desc)
    monza_results.append(r)
    print(f'{desc:45s}  reward={r["total_reward"]:8.2f}  pos={r["final_position"]:2}  rule_violated={r["violated_rule"]}')

In [ ]:
print('\n=== Scenario A — Monza 2025 — Ranked by total reward ===\n')
monza_df = summarise(monza_results)

In [ ]:
# Extract rewards by strategy description
def get_reward(results, desc_fragment):
    for r in results:
        if desc_fragment in r['description']:
            return r['total_reward']
    raise KeyError(desc_fragment)

a1 = get_reward(monza_results, 'S1:')
a2 = get_reward(monza_results, 'S2:')
a3 = get_reward(monza_results, 'S3:')
a4 = get_reward(monza_results, 'S4:')
a5 = get_reward(monza_results, 'S5:')

APPROX_THRESH_A = 5.0  # 4 and 5 are expected approximately tied

print('=== Scenario A Assertions ===')
print(f'  S1={a1:.2f}  S3={a3:.2f}  S4={a4:.2f}  S5={a5:.2f}  S2={a2:.2f}')

checks = [
    ('1 > 3', a1 > a3),
    ('3 > 4', a3 > a4),
    ('3 > 5', a3 > a5),
    ('4 ≈ 5 (|diff| < threshold)', abs(a4 - a5) < APPROX_THRESH_A),
    ('5 > 2', a5 > a2),
    ('4 > 2', a4 > a2),
    ('1 > 2 (rule violator is worst)', a1 > a2),
    ('S2 rule violated', any(r['violated_rule'] for r in monza_results if 'S2' in r['description'])),
]

all_pass = True
for label, result in checks:
    status = 'PASS' if result else 'FAIL'
    if not result:
        all_pass = False
    print(f'  [{status}] {label}')

print(f'\nScenario A overall: {"ALL PASS" if all_pass else "PARTIAL — see FAIL rows above"}')

---
## Scenario B — Bahrain 2024, VER on pole, SOFT start

**Circuit profile:** rival pace ~95 s, rivals pit at lap 12 (VSC-triggered mass cycle).

| Strategy | Pit stops | Expected behaviour |
|---|---|---|
| 1 | S→H lap 18 | 1-stop, HARD durable for rest |
| 2 | S→M lap 18 | 1-stop, MEDIUM has small cliff late |
| 3 | S→M lap 16, M→S lap 38 | 2-stop, tiny SOFT cliff at very end |
| 4 | No pit (stay softs) | Rule violator + massive SOFT cliff abuse |

**Expected assertion: 1 ≈ 2 ≈ 3 > 4** (valid strategies close, soft-only catastrophic)

In [ ]:
BAHRAIN_CFG = {
    'circuit': 'Bahrain Grand Prix',
    'driver': 'VER',
    'year': 2024,
    'total_laps': 57,
    'starting_position': 1,
    'starting_compound': 'SOFT',
    'two_compound_rule_enforced': True,
}

bahrain_strategies = [
    ({18: 'HARD'},              'S1: 1-stop S→H L18'),
    ({18: 'MEDIUM'},            'S2: 1-stop S→M L18'),
    ({16: 'MEDIUM', 38: 'SOFT'}, 'S3: 2-stop S→M→S L16+38'),
    ({},                        'S4: No pit (rule violator + cliff)'),
]

bahrain_results = []
for pit_sched, desc in bahrain_strategies:
    r = run_strategy(env, BAHRAIN_CFG, pit_sched, desc)
    bahrain_results.append(r)
    print(f'{desc:45s}  reward={r["total_reward"]:8.2f}  pos={r["final_position"]:2}  rule_violated={r["violated_rule"]}')

print('\n=== Scenario B — Bahrain 2024 — Ranked by total reward ===\n')
bahrain_df = summarise(bahrain_results)

In [ ]:
b1 = get_reward(bahrain_results, 'S1:')
b2 = get_reward(bahrain_results, 'S2:')
b3 = get_reward(bahrain_results, 'S3:')
b4 = get_reward(bahrain_results, 'S4:')

APPROX_THRESH = 8.0  # within 8 reward points = 'approximately equal'

print('=== Scenario B Assertions ===')
print(f'  S1={b1:.2f}  S2={b2:.2f}  S3={b3:.2f}  S4={b4:.2f}')
print(f'  Approx-equal threshold: ±{APPROX_THRESH}')

checks_b = [
    ('1 ≈ 2 (|diff| < threshold)', abs(b1 - b2) < APPROX_THRESH),
    ('1 ≈ 3 (|diff| < threshold)', abs(b1 - b3) < APPROX_THRESH),
    ('2 ≈ 3 (|diff| < threshold)', abs(b2 - b3) < APPROX_THRESH),
    ('1 > 4 (rule violator worst)', b1 > b4),
    ('2 > 4', b2 > b4),
    ('3 > 4', b3 > b4),
    ('S4 rule violated', any(r['violated_rule'] for r in bahrain_results if 'S4' in r['description'])),
]

all_pass_b = True
for label, result in checks_b:
    status = 'PASS' if result else 'FAIL'
    if not result:
        all_pass_b = False
    print(f'  [{status}] {label}')

print(f'\nScenario B overall: {"ALL PASS" if all_pass_b else "PARTIAL — see FAIL rows above"}')

---
## Scenario C — Monaco 2023, LEC on pole, MEDIUM start

**Note:** Using Monaco 2023 (not 2024) — 2024 data has all stint_number=2 due to red-flagged start.
LEC won Monaco 2023 from pole; scenario is equivalent.

**Circuit profile:** rival pace ~76 s, rivals pit at lap 49 (training median, low-deg late-stop).

| Strategy | Pit stops | Expected behaviour |
|---|---|---|
| 1 | M→H lap 28 | Optimal: pits before cliff, 21 laps before rivals |
| 2 | No pit | Rule violator |
| 3 | Early M→H lap 12 | Early stop, HARD over-ages (21 laps past HARD cliff) |
| 4 | Late M→H lap 60 | Late stop: 28 laps past MEDIUM cliff before pitting |

**Known env limitation:** SandboxRaceEnv uses cumulative-time-based position, so at Monaco
both early and late pitters drop to P20 (static rivals model doesn't capture the real
'starting position ≈ finishing position' procession). Cliff math therefore dominates:
S3 (−23.1 cliff) > S4 (−40.6 cliff) by ~17.5 pts. Fix deferred to Phase 4.5 GridRaceEnv.

**Expected assertion: 1 > 3 > 4 > 2** *(pace_reward provides meaningful per-lap signal
even at Monaco where position_reward ≈ 0)*

In [ ]:
MONACO_CFG = {
    'circuit': 'Monaco Grand Prix',
    'driver': 'LEC',
    'year': 2023,
    'total_laps': 78,
    'starting_position': 1,
    'starting_compound': 'MEDIUM',
    'two_compound_rule_enforced': True,
}

monaco_strategies = [
    ({28: 'HARD'},  'S1: Optimal 1-stop M→H L28'),
    ({},            'S2: No pit (rule violator)'),
    ({12: 'HARD'},  'S3: Early 1-stop M→H L12'),
    ({60: 'HARD'},  'S4: Late 1-stop M→H L60'),
]

monaco_results = []
for pit_sched, desc in monaco_strategies:
    r = run_strategy(env, MONACO_CFG, pit_sched, desc)
    monaco_results.append(r)
    print(f'{desc:45s}  reward={r["total_reward"]:8.2f}  pos={r["final_position"]:2}  rule_violated={r["violated_rule"]}')

print('\n=== Scenario C — Monaco 2023 — Ranked by total reward ===\n')
monaco_df = summarise(monaco_results)

In [ ]:
c1 = get_reward(monaco_results, 'S1:')
c2 = get_reward(monaco_results, 'S2:')
c3 = get_reward(monaco_results, 'S3:')
c4 = get_reward(monaco_results, 'S4:')

print('=== Scenario C Assertions ===')
print(f'  S1={c1:.2f}  S3={c3:.2f}  S4={c4:.2f}  S2={c2:.2f}')

checks_c = [
    ('1 > 3', c1 > c3),
    ('1 > 4', c1 > c4),
    ('3 > 4 (cliff math: S3 ~-23 vs S4 ~-41)', c3 > c4),
    ('3 > 2 (rule violator worst)', c3 > c2),
    ('4 > 2 (rule violator worst)', c4 > c2),
    ('S2 rule violated', any(r['violated_rule'] for r in monaco_results if 'S2' in r['description'])),
]

all_pass_c = True
for label, result in checks_c:
    status = 'PASS' if result else 'FAIL'
    if not result:
        all_pass_c = False
    print(f'  [{status}] {label}')

print(f'  [NOTE] S3 vs S4 |diff|={abs(c3-c4):.2f}: env limitation (cumulative-time position at Monaco).'
      f' Phase 4.5 fix.')
print(f'\nScenario C overall: {"ALL PASS" if all_pass_c else "PARTIAL — see FAIL rows above"}')

---
## Scenario D — Reward Shape Sanity (Monza Optimal 1-Stop)

Plots the per-step reward over the full Monza 2025 optimal 1-stop run (M→H lap 32).

**Expected shape with pace_reward:**
- Laps 1–31: small non-zero signal from `pace_reward` (ego pace vs rival baseline)
- Lap 32: large dip — pit_cost + position loss + negative pace_delta (ego lap = 81+22=103, rival=81.5)
- Lap 34: large positive spike — rival pit lap (rival baseline = 81.5+22=103, ego = ~81.5)
- Laps 35–52: small positive signal from pace_reward + position recovery
- Lap 53: big terminal spike (P1 → −2 + rule → +10)

pace_reward makes the signal **dense** even during flat stay laps — key for Monaco/Hungary.

In [ ]:
# Re-run optimal Monza strategy to get its log
r_opt = run_strategy(env, MONZA_CFG, {32: 'HARD'}, 'Monza optimal 1-stop')
log = r_opt['log']

print(f'Final position: P{r_opt["final_position"]}  Total reward: {r_opt["total_reward"]:.2f}')
print(f'Rule violated: {r_opt["violated_rule"]}')
print(f'\nReward breakdown (all components):')
print(f'  Position reward total:    {log["position_reward"].sum():.3f}')
print(f'  Pit cost total:           {log["pit_cost"].sum():.3f}')
print(f'  Cliff penalty total:      {log["cliff_penalty"].sum():.3f}')
print(f'  Invalid action total:     {log["invalid_action_penalty"].sum():.3f}')
print(f'  Pace reward total:        {log["pace_reward"].sum():.3f}')
print(f'  Terminal pos reward:      {log["terminal_pos_reward"].sum():.3f}')
print(f'  Terminal rule reward:     {log["terminal_rule_reward"].sum():.3f}')
print(f'  Sum = step_reward total:  {log["step_reward"].sum():.3f}')
print(f'\nPace delta stats (rival_time - ego_time):')
non_pit_non_rival = log[(log['pit_cost'] == 0) & (log['lap'] != 34)]
print(f'  Mean pace_delta on stay laps: {log["pace_delta_s"].mean():.3f}s')
print(f'  On pit lap (L32): {log[log["lap"]==32]["pace_delta_s"].values[0]:.3f}s  '
      f'  pace_reward={log[log["lap"]==32]["pace_reward"].values[0]:.3f}')
print(f'  On rival pit lap (L34): {log[log["lap"]==34]["pace_delta_s"].values[0]:.3f}s  '
      f'  pace_reward={log[log["lap"]==34]["pace_reward"].values[0]:.3f}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

laps = log['lap'].astype(int)

# ── top: step reward per lap
ax = axes[0]
ax.bar(laps, log['step_reward'], color=['tab:red' if v < 0 else 'tab:green' for v in log['step_reward']],
       width=0.8, alpha=0.8)
ax.axhline(0, color='white', linewidth=0.5, linestyle='--')
pit_lap = log.loc[log['pit_cost'] < 0, 'lap'].values
for pl in pit_lap:
    ax.axvline(pl, color='yellow', linewidth=1.5, linestyle=':', alpha=0.8, label='ego pit' if pl == pit_lap[0] else '')
ax.axvline(34, color='orange', linewidth=1.5, linestyle=':', alpha=0.8, label='rival pit')
ax.set_ylabel('Step Reward')
ax.set_title('Monza 2025 Optimal 1-Stop (M→H lap 32) — Per-Step Reward (with pace_reward)')
ax.legend(loc='upper left', fontsize=9)
ax.set_facecolor('#111111')

# ── middle: component breakdown (stacked)
ax = axes[1]
base = pd.Series(0.0, index=laps.index)
for col, label, color in [
    ('position_reward', 'position_reward', 'steelblue'),
    ('pace_reward',     'pace_reward',     'mediumseagreen'),
    ('cliff_penalty',   'cliff_penalty',   'tab:orange'),
    ('pit_cost',        'pit_cost',        'tab:purple'),
]:
    ax.bar(laps, log[col], bottom=base, label=label, color=color, alpha=0.85, width=0.8)
    base = base + log[col]
terminal_sum = log['terminal_pos_reward'] + log['terminal_rule_reward']
ax.bar(laps, terminal_sum, bottom=base, label='terminal', color='gold', alpha=0.9, width=0.8)
ax.axhline(0, color='white', linewidth=0.5, linestyle='--')
ax.set_ylabel('Reward Components')
ax.legend(loc='upper left', fontsize=8, ncol=5)
ax.set_facecolor('#111111')

# ── bottom: position over race
ax = axes[2]
ax.plot(laps, log['position_after'], color='tab:cyan', linewidth=2, marker='o', markersize=3)
ax.invert_yaxis()
ax.set_ylabel('Position (lower = better)')
ax.set_xlabel('Lap')
ax.yaxis.set_major_locator(ticker.MultipleLocator(1))
ax.set_facecolor('#111111')

for ax in axes:
    ax.set_facecolor('#111111')
    ax.tick_params(colors='white')
    ax.yaxis.label.set_color('white')
    ax.xaxis.label.set_color('white')
    ax.title.set_color('white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#333333')

fig.patch.set_facecolor('#1a1a1a')
plt.tight_layout()
plt.savefig('../models/figures/reward_shape_monza_optimal.png', dpi=120, bbox_inches='tight',
            facecolor='#1a1a1a')
plt.show()
print('Saved to models/figures/reward_shape_monza_optimal.png')

In [ ]:
# Signal-to-noise: terminal reward vs per-step std
non_terminal = log.iloc[:-1]
step_std = non_terminal['step_reward'].std()
last = log.iloc[-1]
terminal_magnitude = abs(last['terminal_pos_reward'] + last['terminal_rule_reward'])
snr = terminal_magnitude / max(step_std, 0.001)

print('=== Terminal lap breakdown ===')
print(f'  Final position:             P{int(last["position_after"])}')
print(f'  Terminal pos reward:        {last["terminal_pos_reward"]:.2f}')
print(f'  Terminal rule reward:       {last["terminal_rule_reward"]:.2f}')
print(f'  Pace reward on last lap:    {last["pace_reward"]:.3f}')
print(f'  Terminal step total:        {last["step_reward"]:.2f}')
print()
print(f'  Per-step reward std (non-terminal): {step_std:.4f}')
print(f'  Terminal reward magnitude:          {terminal_magnitude:.2f}')
print(f'  SNR (terminal / step std):          {snr:.1f}×')
print(f'  [{"OK" if snr > 3 else "LOW — may be too flat for PPO"}] SNR > 3×')
print()
print('  pace_reward contribution to reward density:')
print(f'  Non-terminal step_reward std WITHOUT pace_reward: '
      f'{(non_terminal["step_reward"] - non_terminal["pace_reward"]).std():.4f}')
print(f'  Non-terminal step_reward std WITH pace_reward:    {step_std:.4f}')

In [ ]:
# Overall summary across all scenarios
print('=' * 60)
print('PHASE 4.2 REWARD FUNCTION VALIDATION SUMMARY')
print('=' * 60)

a_core = a1 > a3 and a3 > a4 and a3 > a5 and a5 > a2
a_tie  = abs(a4 - a5) < 5.0
b_dominates = b1 > b4 and b2 > b4 and b3 > b4
b_close = max(abs(b1-b2), abs(b1-b3), abs(b2-b3)) < 8.0
c_ordered = c1 > c3 > c4 > c2

print(f'  Scenario A (Monza):  1>3>{{4≈5}}>2 = {a_core and a_tie}')
print(f'    rewards: S1={a1:.1f} S3={a3:.1f} S4={a4:.1f} S5={a5:.1f} S2={a2:.1f}')
print(f'    S4 vs S5 |diff|={abs(a4-a5):.1f} (tied within 5-pt threshold: {a_tie})')

print(f'  Scenario B (Bahrain): 1≈2≈3>>4 = {b_dominates and b_close}')
print(f'    rewards: S1={b1:.1f} S2={b2:.1f} S3={b3:.1f} S4={b4:.1f}')
print(f'    max valid spread={max(abs(b1-b2), abs(b1-b3), abs(b2-b3)):.1f} pts')

print(f'  Scenario C (Monaco): 1>3>4>2 = {c_ordered}')
print(f'    rewards: S1={c1:.1f} S3={c3:.1f} S4={c4:.1f} S2={c2:.1f}')
print(f'    S3 vs S4 diff={c3-c4:.1f} (cliff dominates; env limitation — Phase 4.5 fix)')

print(f'  Scenario D (reward shape):')
print(f'    SNR = {snr:.1f}x')
print(f'    pace_reward boosts density: see std comparison above')